# Task 1 — SAM zero-shot (Kaggle, GPU)

SAM con i pesi originali, **senza training**, valutato su validation e test. È il punto di partenza: ogni metodo di fine-tuning deve fare meglio di questi numeri.

**Session options:** *Accelerator* **GPU T4 x2** (ne usiamo una) · *Internet* **On** · *Persistence* **Files only**.

Durata prevista: ~10 minuti, di cui ~2 per i test e ~2 per la valutazione.

## 1. Codice della repo

In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/caMatt99/sam-peft-kvasir.git"
WORK = Path("/kaggle/working")
REPO = WORK / "sam-peft-kvasir"

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}
!pip install -q -e "{REPO}[dev]"
%cd {REPO}
sys.path.insert(0, str(REPO / "src"))   # il kernel già avviato non vede l'install editable
!git log --oneline -1

## 2. Dati e pesi
Kvasir-SEG si scarica dal sito ufficiale; le maschere pulite si rigenerano con `prepare_data.py` (deterministico: sempre lo stesso risultato).
Lo split e i prompt arrivano dalla repo (`data/splits/`).

In [ ]:
import torch
assert torch.cuda.is_available(), "Accendi la GPU in Session options → Accelerator"
print(torch.cuda.get_device_name(0))

def find_kvasir_root():
    for base in [Path("/kaggle/input"), WORK / "data"]:
        if base.exists():
            for p in base.rglob("images"):
                if (p.parent / "masks").is_dir():
                    return p.parent
    return None

RAW = find_kvasir_root()
if RAW is None:
    (WORK / "data").mkdir(parents=True, exist_ok=True)
    !wget -q -L --no-check-certificate https://datasets.simula.no/downloads/kvasir-seg.zip -O /kaggle/working/data/kvasir-seg.zip
    !unzip -q /kaggle/working/data/kvasir-seg.zip -d /kaggle/working/data/ && rm /kaggle/working/data/kvasir-seg.zip
    RAW = find_kvasir_root()
assert RAW is not None

PROCESSED = WORK / "processed"
if not (PROCESSED / "meta.json").exists():
    !python scripts/prepare_data.py --raw-root {RAW} --out {PROCESSED} > /dev/null

CKPT = WORK / "checkpoints" / "sam_vit_b_01ec64.pth"
if not CKPT.exists() or CKPT.stat().st_size < 300e6:
    CKPT.parent.mkdir(parents=True, exist_ok=True)
    !wget -q -L https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {CKPT}

assert Path("data/splits/split.json").exists(), "split.json mancante: è stato fatto il commit di data/splits?"
print(RAW, "|", PROCESSED, "| checkpoint MB:", round(CKPT.stat().st_size / 1e6))

## 3. Test
Tra gli altri, `test_matches_sam_predictor` verifica che il nostro forward dia le stesse maschere di `SamPredictor`.
Se un test fallisce, fermati qui.

In [ ]:
!python -m pytest -q

## 4. Valutazione zero-shot
Per ogni immagine: encoder una volta, poi decoder con i box esatti e con quelli con jitter.
Il numero dopo ± è l'**errore standard della media**: differenze più piccole di ~2 volte questo valore non sono affidabili.

In [ ]:
!python scripts/evaluate.py --raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT} --split val --out runs/zeroshot

In [ ]:
!python scripts/evaluate.py --raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT} --split test --out runs/zeroshot

### Variante: 3 maschere per box
SAM può proporre 3 maschere per ogni box (`multimask_output=True`) con una stima della propria qualità (IoU stimato).
Qui si tiene, per ogni box, quella con l'IoU stimato più alto. Nessun training: è un secondo riferimento zero-shot.
Motivo: nei fallimenti totali della prima esecuzione, una delle 3 maschere era quasi sempre corretta.

In [ ]:
for s in ["val", "test"]:
    !python scripts/evaluate.py --raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT} --split {s} --multimask --out runs/zeroshot_multimask

## 5. Analisi

In [ ]:
import json
import pandas as pd

for split in ["val", "test"]:
    rows = []
    for variant in ["zeroshot", "zeroshot_multimask"]:
        m = json.loads(Path(f"runs/{variant}/{split}/metrics.json").read_text())
        for g, v in m["groups"].items():
            rows.append({"variante": variant, "gruppo": g, "n": v["n"],
                         **{f"{k} {met}": round(v[f"{k}.{met}"]["mean"], 3)
                            for k in ["boxes", "boxes_jitter"] for met in ["dice_raw", "iou_raw"]},
                         **{f"{k} fail": f"{v[f'{k}.fail_rate']:.0%}" for k in ["boxes", "boxes_jitter"]}})
    print(f"\n=== {split} ===  picco memoria GPU: {m['peak_gpu_memory_mb']:.0f} MB")
    display(pd.DataFrame(rows).set_index(["variante", "gruppo"]))

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv("runs/zeroshot/test/per_image.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(df["boxes.dice_raw"], bins=20, range=(0, 1)); ax[0].set_title("Dice per immagine, box esatti")
ax[1].hist(df["boxes_jitter.dice_raw"], bins=20, range=(0, 1)); ax[1].set_title("Dice per immagine, box con jitter")
for a in ax: a.set_xlabel("Dice"); a.set_ylabel("immagini")
plt.show()

print("immagini con Dice < 0.8 (box esatti):", (df["boxes.dice_raw"] < 0.8).sum(), "su", len(df))
print("calo medio dovuto al jitter:", round((df["boxes.dice_raw"] - df["boxes_jitter.dice_raw"]).mean(), 3))

## 6. I casi peggiori del test
Immagine con i box, GT, predizione. Da guardare: **perché** SAM sbaglia? Bordi poco contrastati, riflessi, polipi piatti, box che include altro tessuto…

In [ ]:
import cv2
import numpy as np
from sampeft.data.kvasir import KvasirDataset
from sampeft.model import load_sam, encode, decode
from matplotlib.patches import Rectangle

split = json.loads(Path("data/splits/split.json").read_text())
ds = KvasirDataset(RAW, PROCESSED, split["test"], train=False, prompts_file="data/splits/prompts_test.json")
sam = load_sam(CKPT, device="cuda").eval()
idx = {i: n for n, i in enumerate(ds.ids)}

worst = df.nsmallest(6, "boxes.dice_raw")
for _, row in worst.iterrows():
    x = ds[idx[row["id"]]]
    with torch.no_grad():
        emb = encode(sam, x["image"][None].cuda())
        logits, _ = decode(sam, emb[0], x["boxes"].cuda(), x["input_size"], x["original_size"])
    pred = (logits > 0).any(0).cpu().numpy()
    img = cv2.cvtColor(cv2.imread(str(RAW / "images" / f"{row['id']}.jpg")), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    ax[0].imshow(img)
    for b in x["boxes_orig"].numpy():
        ax[0].add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor="lime", lw=2))
    ax[1].imshow(x["gt"], cmap="gray"); ax[2].imshow(pred, cmap="gray")
    ax[0].set_title(row["id"], fontsize=9); ax[1].set_title("GT"); ax[2].set_title(f"SAM  Dice={row['boxes.dice_raw']:.3f}")
    for a in ax: a.axis("off")
    plt.show()

## 7. Dice e dimensione del polipo
Tutte le 200 immagini di val e test. Il gruppo `large_polyp` (area > 25%) è stato fissato dopo questa analisi e **prima** di qualunque training.

In [ ]:
import numpy as np
from scipy.stats import spearmanr

meta = json.loads((PROCESSED / "meta.json").read_text())
df_all = pd.concat([pd.read_csv(f"runs/{v}/{s}/per_image.csv").assign(split=s, variante=v)
                    for v in ["zeroshot", "zeroshot_multimask"] for s in ["val", "test"]], ignore_index=True)
df_all["bin"] = pd.cut(df_all["area_frac"], [0, .1, .25, .5, 1], labels=["<10%", "10-25%", "25-50%", ">50%"])

tab = (df_all.groupby(["variante", "bin"], observed=True)
       .agg(n=("id", "size"),
            dice=("boxes.dice_raw", "mean"),
            mediana=("boxes.dice_raw", "median"),
            minimo=("boxes.dice_raw", "min"),
            fallimenti=("boxes.dice_raw", lambda s: int((s < 0.5).sum())),
            dice_jitter=("boxes_jitter.dice_raw", "mean")))
display(tab.round(3))

single = df_all[df_all["variante"] == "zeroshot"]
rho, p = spearmanr(single["area_frac"], single["boxes.dice_raw"])
print(f"Spearman(area, Dice), uscita singola: {rho:+.2f}  p = {p:.3f}  n = {len(single)}")

## 8. Diagnosi dei fallimenti (Dice < 0.5, uscita singola)
Per ogni fallimento: immagine con box, GT, e le 3 maschere alternative di SAM con l'IoU stimato e il Dice vero.
Se una delle 3 è buona, SAM "vede" il polipo ma sceglie l'interpretazione sbagliata del box: è un problema del decoder.

In [ ]:
from matplotlib.patches import Rectangle

dsets = {s: KvasirDataset(RAW, PROCESSED, split[s], train=False, prompts_file=f"data/splits/prompts_{s}.json")
         for s in ["val", "test"]}
fails = single[single["boxes.dice_raw"] < 0.5].sort_values("area_frac")
display(fails[["id", "split", "area_frac", "boxes.dice_raw", "boxes_jitter.dice_raw"]].round(3))

for _, row in fails.iterrows():
    d_s = dsets[row["split"]]
    x = d_s[d_s.ids.index(row["id"])]
    gt = np.asarray(x["gt"]).astype(bool)
    with torch.no_grad():
        emb = encode(sam, x["image"][None].cuda())
        sparse, dense = sam.prompt_encoder(points=None, boxes=x["boxes"].cuda(), masks=None)
        low, iou_pred = sam.mask_decoder(image_embeddings=emb, image_pe=sam.prompt_encoder.get_dense_pe(),
                                         sparse_prompt_embeddings=sparse, dense_prompt_embeddings=dense,
                                         multimask_output=True)
        # (K box, 3 maschere, H, W) -> unione sui box -> (3, H, W)
        masks = (sam.postprocess_masks(low, x["input_size"], x["original_size"]) > 0).any(0).cpu().numpy()

    img = cv2.cvtColor(cv2.imread(str(RAW / "images" / f"{row['id']}.jpg")), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 5, figsize=(22, 4))
    ax[0].imshow(img)
    for b in x["boxes_orig"].numpy():
        ax[0].add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor="lime", lw=2))
    ax[0].set_title(f"{row['id']}\n{row['split']} · area {row['area_frac']:.0%}", fontsize=9)
    ax[1].imshow(gt, cmap="gray"); ax[1].set_title(f"GT\nDice uscita singola {row['boxes.dice_raw']:.3f}", fontsize=9)
    for k in range(3):
        d = 2 * (masks[k] & gt).sum() / (masks[k].sum() + gt.sum())
        ax[2 + k].imshow(masks[k], cmap="gray")
        ax[2 + k].set_title(f"maschera {k + 1}\nIoU stimato {iou_pred[:, k].mean():.2f} · Dice {d:.3f}", fontsize=9)
    for a in ax: a.axis("off")
    plt.show()

## 9. Salvataggio
Scarica `zeroshot.zip` (link sotto o pannello Output): `metrics.json` e `per_image.csv` delle due varianti.
**Non va su GitHub**: i numeri li riportiamo nel README.

In [ ]:
from IPython.display import FileLink
!zip -qr zeroshot.zip runs/zeroshot runs/zeroshot_multimask
FileLink("zeroshot.zip")   # percorso relativo alla cartella corrente (la repo): niente cambi di cartella

## Da annotare
1. Dice medio sul test con box esatti e con jitter (± errore standard).
2. Il Dice sulla GT raw e su quella pulita coincide? (atteso: sì)
3. Tempo dell'encoder e picco di memoria GPU.
4. Nei casi peggiori, che tipo di errore fa SAM?
5. Con 3 maschere per box i fallimenti spariscono? E la media generale migliora o peggiora?